# NYC HVFHV — Data Preprocessing

Applies data quality filters and type conversions to the raw HVFHV trip data 
(see `01_data_understanding.ipynb` for the EDA and rationale behind each step), 
producing a clean table (`fhvhv_clean`) for downstream hypothesis testing.

**Input:** `fhvhv_raw_2026` (materialized from `../data/raw/fhvhv_tripdata_2026-*.parquet`)

**Reference Data:** `../data/reference/taxi_zone_lookup.csv`

**Output:** `fhvhv_clean_2026`

## Preprocessing Steps Applied
- Removed `trip_miles <= 0` (~11.5K rows, 0.01%)
- Removed invalid trip duration — `dropoff_datetime <= pickup_datetime` (2 rows)
- Removed `base_passenger_fare <= 0` (0.14%)
- Removed `driver_pay <= 0` (0.06%)
- Removed timestamp discrepancies — `on_scene_datetime` before `request_datetime` (~1.9M rows, 1.8%) 
  and `on_scene_datetime` after `pickup_datetime` (~3.2K rows, 0.003%)
- Converted Y/N flags to booleans — `shared_request_flag`, `shared_match_flag`, 
  `access_a_ride_flag`, `wav_request_flag`, `wav_match_flag`
- Mapped provider codes to names — HV0003 → Uber, HV0005 → Lyft
- Joined the NYC Taxi Zone Lookup table to add pickup and dropoff zone, borough, and service zone information

In [14]:
import duckdb
from nyc_fhvhv.cleaning import clean_fhvhv

con = duckdb.connect("../data/nyc_fhvhv_2026.duckdb")

# Our source of truth is the raw fhvhv data
con.execute("""
CREATE OR REPLACE TABLE fhvhv_raw_2026 AS
SELECT DISTINCT *
FROM read_parquet('../data/raw/fhvhv_tripdata_2026-*.parquet');

CREATE OR REPLACE TABLE zone_lookup AS
SELECT * FROM read_csv('../data/reference/taxi_zone_lookup.csv');
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [15]:
con.sql("""
SELECT COUNT(*) FROM fhvhv_raw_2026;
""").df()

,count_star()
0,105996113


In [16]:
con.sql("""
SELECT COUNT(*) FROM zone_lookup;
""").df()

,count_star()
0,265


In [17]:
con.sql("""
SELECT 
    * 
FROM zone_lookup
LIMIT 5;
""").df()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [18]:
# Clean the FHVHV data and store it in a new table
clean_fhvhv(
    con,
    raw_table="fhvhv_raw_2026",
    clean_table="fhvhv_clean_2026"
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

'fhvhv_clean_2026'

In [19]:
con.sql("""
SELECT
    *
FROM fhvhv_clean_2026
LIMIT 5;
""").df()

,provider_name,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,pickup_borough,pickup_zone,...,congestion_surcharge,airport_fee,tips,driver_pay,cbd_congestion_fee,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,Uber,B03404,B03404,2026-05-23 22:51:00,2026-05-23 22:56:53,2026-05-23 22:57:03,2026-05-23 23:08:33,16,Queens,Bayside,...,0.00,0.0,3.59,14.24,0.0,False,False,False,False,False
1,Lyft,B03406,NaN,2026-05-23 22:26:32,2026-05-23 22:30:46,2026-05-23 22:32:16,2026-05-23 22:41:49,206,Staten Island,Saint George/New Brighton,...,0.00,0.0,0.00,9.01,0.0,False,False,False,False,False
2,Uber,B03404,B03404,2026-05-23 22:39:24,2026-05-23 22:47:12,2026-05-23 22:47:44,2026-05-23 23:02:43,148,Manhattan,Lower East Side,...,2.75,0.0,0.00,13.24,1.5,False,False,False,False,False
3,Uber,B03404,B03404,2026-05-23 22:39:09,2026-05-23 22:41:38,2026-05-23 22:43:39,2026-05-23 22:53:13,89,Brooklyn,Flatbush/Ditmas Park,...,0.00,0.0,0.00,9.11,0.0,False,False,False,False,False
4,Lyft,B03406,NaN,2026-05-23 22:03:21,2026-05-23 22:06:57,2026-05-23 22:07:13,2026-05-23 22:16:33,94,Bronx,Fordham South,...,0.00,0.0,0.00,8.36,0.0,False,False,False,False,False


In [20]:
con.sql("""
DESCRIBE fhvhv_clean_2026;
""").df()

,column_name,column_type,null,key,default,extra
0,provider_name,VARCHAR,YES,None,None,None
1,dispatching_base_num,VARCHAR,YES,None,None,None
2,originating_base_num,VARCHAR,YES,None,None,None
3,request_datetime,TIMESTAMP,YES,None,None,None
4,on_scene_datetime,TIMESTAMP,YES,None,None,None
5,pickup_datetime,TIMESTAMP,YES,None,None,None
6,dropoff_datetime,TIMESTAMP,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,pickup_borough,VARCHAR,YES,None,None,None
9,pickup_zone,VARCHAR,YES,None,None,None


In [21]:
con.sql("""
SELECT COUNT(*) FROM fhvhv_clean_2026;
""").df()

,count_star()
0,103921454


In [22]:
# Inspect the pickup borough distribution
con.sql("""
SELECT 
    pickup_borough,
    pickup_zone,
    COUNT(*) AS pickup_count
FROM fhvhv_clean_2026
GROUP BY pickup_borough, pickup_zone
ORDER BY pickup_borough, pickup_zone
""").df()

,pickup_borough,pickup_zone,pickup_count
0,Bronx,Allerton/Pelham Gardens,254584
1,Bronx,Bedford Park,518312
2,Bronx,Belmont,326125
3,Bronx,Bronx Park,40803
4,Bronx,Bronxdale,273224
...,...,...,...
256,Staten Island,Saint George/New Brighton,234097
257,Staten Island,South Beach/Dongan Hills,96121
258,Staten Island,Stapleton,158107
259,Staten Island,West Brighton,89216


In [23]:
con.sql("""
SELECT
    SUM(trip_miles <= 0) AS invalid_trip_miles,
    SUM(dropoff_datetime <= pickup_datetime) AS invalid_duration,
    SUM(base_passenger_fare <=0) AS invalid_fare,
    SUM(driver_pay <=0) AS invalid_driver_pay,
    SUM(on_scene_datetime < request_datetime) AS invalid_request_time,
    SUM(on_scene_datetime > pickup_datetime) AS invalid_pickup_time
FROM fhvhv_clean_2026;
""").df()

,invalid_trip_miles,invalid_duration,invalid_fare,invalid_driver_pay,invalid_request_time,invalid_pickup_time
0,0.0,0.0,0.0,0.0,0.0,0.0


In [24]:
con.sql("""
SELECT 
    provider_name,
    COUNT(*) AS total_trips,
    SUM(CASE WHEN originating_base_num IS NULL THEN 1 ELSE 0 END) AS nulls
FROM fhvhv_clean_2026
GROUP BY provider_name;
""").df()

,provider_name,total_trips,nulls
0,Lyft,29310514,29266702.0
1,Uber,74610940,0.0


In [25]:
con.sql("""
SELECT
    COUNT(*) AS total_trips,
    SUM(CASE WHEN base_passenger_fare <= 0 THEN 1 ELSE 0 END) AS invalid_base_passenger_fare_count,
    SUM(CASE WHEN driver_pay <= 0 THEN 1 ELSE 0 END) AS invalid_driver_pay_count,
    (SUM(CASE WHEN base_passenger_fare <= 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*)) AS invalid_base_passenger_fare_pct,
    (SUM(CASE WHEN driver_pay <= 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*)) AS invalid_driver_pay_pct
FROM fhvhv_clean_2026;
""").df()

,total_trips,invalid_base_passenger_fare_count,invalid_driver_pay_count,invalid_base_passenger_fare_pct,invalid_driver_pay_pct
0,103921454,0.0,0.0,0.0,0.0


In [26]:
con.close()